In [1]:
import warnings
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pyspark.sql.dataframe
from datetime import datetime
from IPython.display import display
import pandas as pd
import sys
import os
import json
#sys.path.append(os.path.abspath(".."))
#from utils.kafka_config import KafkaConfig
import yaml
from pathlib import Path
# Configure pandas to show ful output without truncation
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)     # Show all rows
pd.set_option('display.max_colwidth', None) # Don't truncate columns content
pd.set_option('display.width', None)        # Use full width

print(" Libraries imported successfully")
warnings.filterwarnings("ignore")

 Libraries imported successfully


In [2]:
try: 
    from pyspark import SparkContext
    sc = SparkContext._active_spark_context
    if sc: 
        sc.stop()
        print(" Stoped previous SparkContext")
except:
    pass
    
spark = (
    SparkSession
    .builder
    .appName("Streaming from Kafka")
    .config("spark.streaming.stopGracefullyOnShutdown", True)
    .config("spark.sql.shuffle.partitions", 4)
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") 
    .config("spark.sql.catalog.lake",
            "org.apache.iceberg.spark.SparkCatalog") 
    .config("spark.sql.catalog.lake.type", "hadoop") 
    .config("spark.sql.catalog.lake.warehouse", "s3a://lake/warehouse") 
    .config("spark.ssl.enabled", "false")
   .master("local[*]")
    .getOrCreate()
)
spark

In [3]:
def print_as_df(df, limit = 5):
    if isinstance(df, pyspark.sql.dataframe.DataFrame):
         display(df.limit(limit).toPandas())
    else:
        print('Unknow types. Just spark dataframe is acceptable')

In [4]:
def get_last_watermark(table_name:str, layer:str, prev_layer:str) -> datetime:
    last_watermark = datetime(2026, 5, 1) # default date
    try: 
        last_watermark = spark.sql(f"""
        SELECT COALESCE(
            (SELECT last_watermark FROM lake.{layer}.watermarks WHERE table_name = '{table_name}'),
            (SELECT MIN(ingestion_ts) FROM lake.{prev_layer}.{table_name})
        ) as last_watermark
    """).collect()[0][0]
        return last_watermark
    except Exception as e : 
        print(e, "\ndefault water_mark",last_watermark)
        raise

In [5]:
def flatten_kafka_payload(df:pyspark.sql.dataframe.DataFrame, table_schema):
     #if isinstance(df, pyspark.sql.dataframe.DataFrame):
     return df.withColumn(
        "after_payload_value", from_json(col("after_payload"),table_schema)
                        ).select(
                                "*",
                                "after_payload_value.*",    
                            ).filter(
                                    (col("operation") != 'd') & (col("after_payload").isNotNull() 
                                                                )).drop("after_payload_value", "after_payload", "before_payload")
     

In [6]:
def fetch_bronze_layer_data(table_path_name, water_mark):
    return spark.sql(f""" SELECT * 
                          FROM {table_path_name}
                          WHERE ingestion_ts >= CAST('{water_mark}' AS TIMESTAMP)""")

In [7]:
def update_watermark_table(table_name, layer_path = "lake.silver"):
    try:
        spark.sql(f"""
                    MERGE INTO {layer_path}.watermarks w
                    USING (SELECT '{table_name}' as table_name, CURRENT_TIMESTAMP as last_watermark, CURRENT_TIMESTAMP as updated_at) src
                    ON w.table_name = src.table_name
                    WHEN MATCHED THEN UPDATE SET w.last_watermark = src.last_watermark, w.updated_at = CURRENT_TIMESTAMP
                    WHEN NOT MATCHED THEN INSERT *
                    """)
        print(f"Updating the watermark table {layer_path}.{table_name} Successed ")
    except Exception as e:
        raise Exception(f"Could not update the water mark for {table_name}\n", e)

In [8]:
def run_func(func, *args, **kwargs):
    start_time = datetime.now()

    func(*args, **kwargs)

    end_time = datetime.now()

    duration = (end_time - start_time).total_seconds()

    print(f"Duration: {duration} sec")

In [16]:
print_as_df(spark.sql(""" SHOW TABLES IN lake.gold_dbt_test__audit;
                    """), limit = 2
           )
print_as_df(spark.sql("""SELECT order_id, restaurant_id, placed_at_ts, restaurant_key
FROM lake.gold.fact_orders
WHERE restaurant_key IS NULL;
                    """), limit = 29
           )

,namespace,tableName,isTemporary
0,gold_dbt_test__audit,accepted_values_dim_drivers_a3597ce8cda80b564b670861008ed32a,False
1,gold_dbt_test__audit,dbt_utils_accepted_range_dim_menu_items_price__True__0,False


AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `restaurant_id` cannot be resolved. Did you mean one of the following? [`restaurant_key`, `status`, `created_at_ts`, `customer_id`, `discount`].; line 1 pos 17;
'Project [order_id#312, 'restaurant_id, placed_at_ts#335, restaurant_key#314]
+- Filter isnull(restaurant_key#314)
   +- SubqueryAlias lake.gold.fact_orders
      +- RelationV2[order_key#311, order_id#312, customer_id#313, restaurant_key#314, driver_key#315, pickup_zone_key#316, dropoff_zone_key#317, placed_date_key#318, delivered_date_key#319, placed_time_key#320, delivered_time_key#321, status#322, subtotal#323, delivery_fee#324, service_fee#325, discount#326, tip#327, total#328, prep_duration_min#329, pickup_wait_min#330, delivery_duration_min#331, total_fulfillment_min#332, is_delivered#333, is_cancelled#334, ... 9 more fields] lake.gold.fact_orders lake.gold.fact_orders


In [56]:
print_as_df(spark.sql(""" SELECT * FROM  lake.silver.orders ord
LEFT JOIN lake.gold.dim_restaurants r
   ON  ord.restaurant_id = r.restaurant_id
   AND ord.placed_at_ts >= r.eff_start_ts
   AND ord.placed_at_ts <  COALESCE(r.eff_end_ts, TIMESTAMP '9999-12-31')
   WHERE order_id = 1
"""), limit = 5)

# placed_at_ts 2026-05-12 20:10:12.074
# eff_start_ts 2026-05-12 20:07:56.477
# eff_end_ts NaT

,order_id,customer_id,restaurant_id,driver_id,pickup_zone_id,dropoff_zone_id,status,subtotal,delivery_fee,service_fee,discount,tip,total,placed_at_ts,confirmed_at_ts,ready_at_ts,picked_up_at_ts,delivered_at_ts,cancelled_at_ts,created_at_ts,last_source_update_ts,last_refresh_ts,restaurant_key,restaurant_id,name,cuisine_type,city_id,zone_id,address,rating_avg,is_active,is_current,eff_start_ts,eff_end_ts,onboarded_at,created_at_ts,last_source_update_ts,last_refresh_ts
0,1,4757,2576,None,46,51,cancelled,44,9,4,0,None,57,2026-05-12 20:10:11.764,2026-05-12 20:11:07.582,NaT,NaT,NaT,2026-05-12 20:13:55.211,2026-05-12 20:10:11.764,2026-05-12 20:13:55.211,2026-06-22 22:31:21.671439,bb7404e268dbe1d7b5377ddb6543672484bf198c7e5f573af7e8c8a5e7f5821d,2576,Koshary Abu Tarek - Branch 39,egyptian,4,46,"6854 Vanessa Parks, Medina, Saudi Arabia",4,True,True,2026-05-12 20:07:56.477,NaT,2024-11-30 04:59:47.999,2026-05-12 20:07:56.477,2026-05-12 20:07:56.477,2026-06-27 02:11:10.340411


# lake.silver.orders processing 

In [225]:
# {"order_id":2085275,"customer_id":38246,"restaurant_id":2484,"driver_id":null,"pickup_zone_id":61,"dropoff_zone_id":56,"status":"placed","subtotal":481.77,"delivery_fee":5.11,"service_fee":6.35,"discount":0.0,"tip":0.0,"total":493.23,"placed_at":"2026-05-17T22:05:02.882216Z","confirmed_at":null,"ready_at":null,"picked_up_at":null,"delivered_at":null,"cancelled_at":null,"created_at":"2026-05-17T22:05:02.882398Z","updated_at":"2026-05-17T22:05:02.882398Z"}

In [233]:
# orders_schema = StructType([
#     StructField("order_id", LongType(), False),
#     StructField("customer_id", LongType(), False),
#     StructField("restaurant_id", LongType(), False),
#     StructField("driver_id", LongType(), True),
#     StructField("pickup_zone_id", LongType(), False),
#     StructField("dropoff_zone_id", LongType(), False),
#     StructField("status", StringType(), False),
#     StructField("subtotal", FloatType(), False),
#     StructField("delivery_fee", FloatType(), False),
#     StructField("service_fee", FloatType(), False),
#     StructField("discount", FloatType(), False),
#     StructField("tip", FloatType(), False),
#     StructField("total", FloatType(), False),
#     StructField("placed_at", TimestampType(), False),
#     StructField("confirmed_at", TimestampType(), True),
#     StructField("ready_at", TimestampType(), True),
#     StructField("picked_up_at", TimestampType(), True),
#     StructField("delivered_at", TimestampType(), True),
#     StructField("cancelled_at", TimestampType(), True),
#     StructField("created_at", TimestampType(), False),
#     StructField("updated_at", TimestampType(), False),

# ])

In [65]:
spark.sql("""
        CREATE NAMESPACE IF NOT EXISTS lake.silver;
        """)

DataFrame[]

In [93]:
spark.sql("""
        CREATE OR REPLACE TABLE  lake.silver.orders (
    order_id         STRING,
    customer_id      STRING,
    restaurant_id    STRING,
    driver_id        STRING,
    pickup_zone_id   STRING,
    dropoff_zone_id  STRING,
    status           STRING,
    subtotal         DECIMAL ,
    delivery_fee     DECIMAL,
    service_fee      DECIMAL,
    discount         DECIMAL,
    tip              DECIMAL,
    total            DECIMAL,
    placed_at_ts        TIMESTAMP,
    confirmed_at_ts     TIMESTAMP,
    ready_at_ts         TIMESTAMP,
    picked_up_at_ts     TIMESTAMP,
    delivered_at_ts     TIMESTAMP,
    cancelled_at_ts     TIMESTAMP,
    created_at_ts       TIMESTAMP,
    last_source_update_ts TIMESTAMP,
    last_refresh_ts TIMESTAMP
        )
        USING iceberg
        PARTITIONED BY (days(placed_at_ts))
        TBLPROPERTIES (
            'format-version'                  = '2',
            'write.format.default'            = 'parquet',
            'write.parquet.compression-codec' = 'zstd',
            'write.target-file-size-bytes'    = '134217728'
        );
        """)

DataFrame[]

In [94]:
orders_schema = StructType([
    StructField("order_id", LongType(), False),
    StructField("customer_id", LongType(), False),
    StructField("restaurant_id", LongType(), False),
    StructField("driver_id", LongType(), True),
    StructField("pickup_zone_id", LongType(), False),
    StructField("dropoff_zone_id", LongType(), False),
    StructField("status", StringType(), False),  
    StructField("subtotal", DecimalType(10,2), False),
    StructField("delivery_fee", DecimalType(8,2), False),
    StructField("service_fee", DecimalType(8,2), False),
    StructField("discount", DecimalType(8,2), False),
    StructField("tip", DecimalType(8,2), False),
    StructField("total", DecimalType(10,2), False),
    StructField("placed_at", TimestampType(), False),
    StructField("confirmed_at", TimestampType(), True),
    StructField("ready_at", TimestampType(), True),
    StructField("picked_up_at", TimestampType(), True),
    StructField("delivered_at", TimestampType(), True),
    StructField("cancelled_at", TimestampType(), True),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False),

])

In [97]:
def transforming_orders_to_silver(json_schema, table_name = "orders"):

    print(f"Start transforming \ntable name: {table_name}, table type: FACT, CDC? Yes, transforming type: UPSERT")
    try:
        last_watermark = get_last_watermark(table_name,
                                            layer='silver',
                                            prev_layer='bronze'
                                           )
        
        df_row_data_bronze_layer = fetch_bronze_layer_data(table_path_name = "lake.bronze."+table_name,
                                                            water_mark=last_watermark
                                                          )
        
        df_flattened = flatten_kafka_payload(df = df_row_data_bronze_layer, 
                                             table_schema = json_schema)
        window = Window.partitionBy("order_id").orderBy(col("source_lsn").desc())
    
        # remove duplicate rows 
        df_flattened_partitioned = df_flattened.withColumn("rn", row_number().over(window)).filter(col("rn") == 1).drop('rn')

        if df_flattened_partitioned.isEmpty():
            print(f"the bronze layer data is Empty!! for table {table_name} last water mark {last_watermark}")
            return 
        df_flattened_partitioned = df_flattened_partitioned.select(
                    "order_id",
                    "customer_id",
                    "restaurant_id",
                    "driver_id",
                    "pickup_zone_id",
                    "dropoff_zone_id",
                    "status",
                    "subtotal" ,
                    "delivery_fee",
                    "service_fee",
                    "discount",  
                    "tip",
                    "total",
                       col("placed_at").alias("placed_at_ts"), 
                        col("confirmed_at").alias("confirmed_at_ts"),
                        col("ready_at").alias("ready_at_ts"),
                        col("picked_up_at").alias("picked_up_at_ts"),
                        col("delivered_at").alias("delivered_at_ts"),
                        col("cancelled_at").alias("cancelled_at_ts"),
                        col("created_at").alias("created_at_ts"),
                        col("updated_at").alias("updated_at_ts")
        )

        df_flattened_partitioned.createOrReplaceTempView("silver_orders_updates")
        source = spark.table("silver_orders_updates").alias("s")
        target = spark.table("lake.silver.orders").alias("t")
        updates = (
            source
            .join(target, on="order_id", how="left")
            .withColumn(
                "action",
                when(col("t.order_id").isNotNull(), "update")
                .otherwise("insert")
            )
        )

        # if the load is havy use .cache() then .persist to release  the memory 
        updates.groupBy("action").count().show()
        spark.sql("""
            MERGE INTO lake.silver.orders od
            USING silver_orders_updates s
            ON od.order_id = s.order_id
            
            WHEN MATCHED THEN UPDATE SET
                od.customer_id           = s.customer_id,
                od.restaurant_id         = s.restaurant_id,
                od.driver_id             = s.driver_id,
                od.pickup_zone_id        = s.pickup_zone_id,
                od.dropoff_zone_id       = s.dropoff_zone_id,
                od.status                = s.status,
                od.subtotal              = s.subtotal,
                od.delivery_fee          = s.delivery_fee,
                od.service_fee           = s.service_fee,
                od.discount              = s.discount,
                od.tip                   = s.tip,
                od.total                 = s.total,
                od.placed_at_ts          = s.placed_at_ts,
                od.confirmed_at_ts       = s.confirmed_at_ts,
                od.ready_at_ts           = s.ready_at_ts,
                od.picked_up_at_ts       = s.picked_up_at_ts,
                od.delivered_at_ts       = s.delivered_at_ts,
                od.cancelled_at_ts       = s.cancelled_at_ts,
                od.created_at_ts         = s.created_at_ts,
                od.last_source_update_ts = s.updated_at_ts,        
                od.last_refresh_ts  = CURRENT_TIMESTAMP
            
            WHEN NOT MATCHED THEN INSERT (
                order_id,
                customer_id,
                restaurant_id,
                driver_id,
                pickup_zone_id,
                dropoff_zone_id,
                status,
                subtotal,
                delivery_fee,
                service_fee,
                discount,
                tip,
                total,
                placed_at_ts,
                confirmed_at_ts,
                ready_at_ts,
                picked_up_at_ts,
                delivered_at_ts,
                cancelled_at_ts,
                created_at_ts,
                last_source_update_ts,
                last_refresh_ts
            )
            VALUES (
                s.order_id,
                s.customer_id,
                s.restaurant_id,
                s.driver_id,
                s.pickup_zone_id,
                s.dropoff_zone_id,
                s.status,
                s.subtotal,
                s.delivery_fee,
                s.service_fee,
                s.discount,
                s.tip,
                s.total,
                s.placed_at_ts,
                s.confirmed_at_ts,
                s.ready_at_ts,
                s.picked_up_at_ts,
                s.delivered_at_ts,
                s.cancelled_at_ts,
                s.created_at_ts,
                s.updated_at_ts,
                CURRENT_TIMESTAMP          -- last_refresh_date_ts
            )
            """)
        print("MERGE orders data from bronze --> silver: successed")
        update_watermark_table(table_name = table_name)
        print("  water mark have been updated will be >", last_watermark)
    except Exception as e : 
        raise Exception(e + "\n transforming orders from bronze --> silver: failed")

In [98]:
run_func(transforming_orders_to_silver, orders_schema)

Start transforming 
table name: orders, table type: FACT, CDC? Yes, transforming type: UPSERT
the bronze layer data is Empty!! for table orders last water mark 2026-06-22 22:31:49.340831
Duration: 0.490851 sec


In [ ]:
print_as_df(spark.sql("select * from lake.silver.orders"))

In [62]:
spark.sql("DELETE FROM lake.silver.watermarks WHERE table_name = 'orders'")

DataFrame[]

# lake.silver.order_items processing  

In [227]:
df_order_items = spark.sql("SELECT * FROM lake.bronze.order_items")
print_as_df(df_order_items)

,operation,source_ts_ms,source_lsn,source_txid,source_table,source_snapshot,before_payload,after_payload,kafka_topic,kafka_partition,kafka_offset,kafka_timestamp,ingestion_ts
0,c,1779054968552,30266828768,6012762,order_items,false,None,"{""order_item_id"":6121123,""order_id"":2074054,""menu_item_id"":16288,""quantity"":3,""unit_price"":13.65,""line_total"":40.95,""created_at"":""2026-05-17T21:56:08.548746Z""}",cdc.public.order_items,1,7083,2026-05-17 21:56:09.498,2026-05-17 21:56:40.610
1,c,1779054968552,30266837664,6012762,order_items,false,None,"{""order_item_id"":6121126,""order_id"":2074054,""menu_item_id"":16292,""quantity"":2,""unit_price"":17.37,""line_total"":34.74,""created_at"":""2026-05-17T21:56:08.548746Z""}",cdc.public.order_items,1,7084,2026-05-17 21:56:09.498,2026-05-17 21:56:40.610
2,c,1779054968599,30266858144,6012763,order_items,false,None,"{""order_item_id"":6121129,""order_id"":2074055,""menu_item_id"":13169,""quantity"":2,""unit_price"":32.5,""line_total"":65.0,""created_at"":""2026-05-17T21:56:08.593853Z""}",cdc.public.order_items,1,7085,2026-05-17 21:56:09.498,2026-05-17 21:56:40.610
3,c,1779054968733,30266883888,6012765,order_items,false,None,"{""order_item_id"":6121134,""order_id"":2074057,""menu_item_id"":13807,""quantity"":3,""unit_price"":25.9,""line_total"":77.7,""created_at"":""2026-05-17T21:56:08.729412Z""}",cdc.public.order_items,1,7086,2026-05-17 21:56:09.498,2026-05-17 21:56:40.610
4,c,1779054968780,30266892680,6012766,order_items,false,None,"{""order_item_id"":6121135,""order_id"":2074058,""menu_item_id"":14726,""quantity"":1,""unit_price"":3.0,""line_total"":3.0,""created_at"":""2026-05-17T21:56:08.775655Z""}",cdc.public.order_items,1,7087,2026-05-17 21:56:09.498,2026-05-17 21:56:40.610


In [310]:
spark.sql("""
        CREATE OR REPLACE TABLE lake.silver.order_items (
            order_item_id         STRING,
            order_id              STRING,
            menu_item_id          STRING,
            quantity              INT,
            unit_price            DECIMAL,
            line_total            DECIMAL,
            created_at            TIMESTAMP,
            last_source_update_ts TIMESTAMP,
            last_refresh_date_ts TIMESTAMP
        )
        USING iceberg
        PARTITIONED BY (days(created_at))
        TBLPROPERTIES (
            'format-version'                  = '2',
            'write.format.default'            = 'parquet',
            'write.parquet.compression-codec' = 'zstd',
            'write.target-file-size-bytes'    = '134217728'
        );
        """)

DataFrame[]

In [ ]:
#{"order_item_id":6121123,"order_id":2074054,"menu_item_id":16288,"quantity":3,"unit_price":13.65,"line_total":40.95,"created_at":"2026-05-17T21:56:08.548746Z"}

In [235]:
# order_itmes_schema = StructType([
#     StructField("order_item_id", LongType(), False),
#     StructField("order_id", LongType(), False),
#     StructField("menu_item_id", LongType(), False),
#     StructField("quantity", IntegerType(), False),
#     StructField("unit_price", FloatType(), False),
#     StructField("line_total", FloatType(), False),
#     StructField("created_at", TimestampType(), False),
#     StructField("updated_at", TimestampType(), False),

# ])

In [372]:
df_order_items = spark.sql("SELECT count(*) FROM lake.silver.order_items")
print_as_df(df_order_items)

,count(1)
0,78584


In [311]:
order_itmes_schema = StructType([
    StructField("order_item_id", LongType(), False),
    StructField("order_id", LongType(), False),
    StructField("menu_item_id", LongType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("unit_price", FloatType(), False),
    StructField("line_total", FloatType(), False),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False),

])
def transforming_order_items_to_silver(order_itmes_schema, table_name='order_items'):

    print(f"Start transforming\ntable name: {table_name}, table type: FACT, CDC? Yes, transforming type: APPEND")
    try:    
        
        last_watermark = get_last_watermark(table_name,
                                            layer='silver',
                                            prev_layer='bronze'
                                           )
    
        df_row_data_bronze_layer = fetch_bronze_layer_data(table_path_name = "lake.bronze."+table_name,
                                                            water_mark=last_watermark
                                                          )
        df_order_items_flatern = flatten_kafka_payload(df = df_row_data_bronze_layer,
                                                       table_schema = order_itmes_schema ).dropDuplicates(['order_item_id', 'source_lsn'])

        df_order_items_ids = spark.sql("""
                    SELECT  oditm.order_item_id  
                    FROM lake.silver.order_items oditm
                 """
                 )
        df_order_items_joined =  df_order_items_flatern.join(
                                        df_order_items_ids,
                                        on="order_item_id", how='left_anti'
                                        )
        print("  Is there data in the silver layer?",not df_order_items_ids.isEmpty())
        print("  Is there data after left join the bronze layer with silver layer?",not df_order_items_joined.isEmpty())
        if df_order_items_joined.isEmpty():
            print(f"the bronze layer data is Empty!! for table {table_name} last water mark {last_watermark}")
            return 
        df_order_items_joined = df_order_items_joined.withColumnRenamed("updated_at", "last_source_update_ts")
        df_order_items_joined = df_order_items_joined.withColumn("last_refresh_date_ts", current_timestamp())
        df_order_items_joined = df_order_items_joined.select(
                                        "order_item_id",
                                        "order_id",
                                        "menu_item_id",
                                        "quantity",
                                        "unit_price",
                                        "line_total",
                                        "created_at"  ,
                                        "last_source_update_ts",
                                        "last_refresh_date_ts" 
        )
        df_order_items_joined.createOrReplaceTempView("order_items_apending")
        count_new_rows = df_order_items_joined.count()

        spark.sql("""
                  INSERT INTO lake.silver.order_items 
                  SELECT * FROM order_items_apending
                  """)
        print(f"APPEND {table_name} data from bronze --> silver: successed")
        print("  total of new inserts",count_new_rows)
        update_watermark_table(table_name = table_name)
        print("  water mark have been updated will be >", last_watermark)
    except Exceptiona as e : 
        raise Exception(e + f"\n transforming {table_name} from bronze --> silver: failed")
        

In [313]:
transforming_order_items_to_silver(order_itmes_schema)

Start transforming
table name: order_items, table type: FACT, CDC? Yes, transforming type: APPEND
  Is there data in the silver layer? True
  Is there data after left join the bronze layer with silver layer? False
  No data to insert


# lake.silver.payments processing

In [230]:
df_payments = spark.sql("SELECT * FROM lake.bronze.payments")
print_as_df(df_payments)

,operation,source_ts_ms,source_lsn,source_txid,source_table,source_snapshot,before_payload,after_payload,kafka_topic,kafka_partition,kafka_offset,kafka_timestamp,ingestion_ts
0,c,1778948691874,30107328208,5996463,payments,false,None,"{""payment_id"":2070813,""order_id"":2070813,""payment_method"":""cash"",""amount"":113.29,""status"":""pending"",""processor_ref"":null,""paid_at"":null,""created_at"":""2026-05-16T16:24:51.870969Z"",""updated_at"":""2026-05-16T16:24:51.870969Z""}",cdc.public.payments,2,3103,2026-05-16 16:24:52.729,2026-05-16 16:25:20.751
1,u,1778948691896,30107329768,5996465,payments,false,"{""payment_id"":2070686,""order_id"":2070686,""payment_method"":""card"",""amount"":441.33,""status"":""pending"",""processor_ref"":null,""paid_at"":null,""created_at"":""2026-05-16T16:24:46.012176Z"",""updated_at"":""2026-05-16T16:24:46.012176Z""}","{""payment_id"":2070686,""order_id"":2070686,""payment_method"":""card"",""amount"":441.33,""status"":""captured"",""processor_ref"":""ch_35zgdlr9fjwrkjgqv7y87uzc"",""paid_at"":""2026-05-16T16:24:51.895432Z"",""created_at"":""2026-05-16T16:24:46.012176Z"",""updated_at"":""2026-05-16T16:24:51.895558Z""}",cdc.public.payments,2,3104,2026-05-16 16:24:52.729,2026-05-16 16:25:20.751
2,u,1778948691897,30107330376,5996466,payments,false,"{""payment_id"":2070687,""order_id"":2070687,""payment_method"":""card"",""amount"":218.96,""status"":""pending"",""processor_ref"":null,""paid_at"":null,""created_at"":""2026-05-16T16:24:46.057002Z"",""updated_at"":""2026-05-16T16:24:46.057002Z""}","{""payment_id"":2070687,""order_id"":2070687,""payment_method"":""card"",""amount"":218.96,""status"":""captured"",""processor_ref"":""ch_anbxs375d309bsydvvrtp56o"",""paid_at"":""2026-05-16T16:24:51.897301Z"",""created_at"":""2026-05-16T16:24:46.057002Z"",""updated_at"":""2026-05-16T16:24:51.897416Z""}",cdc.public.payments,2,3105,2026-05-16 16:24:52.729,2026-05-16 16:25:20.751
3,u,1778948691900,30107330984,5996467,payments,false,"{""payment_id"":2070688,""order_id"":2070688,""payment_method"":""card"",""amount"":300.0,""status"":""pending"",""processor_ref"":null,""paid_at"":null,""created_at"":""2026-05-16T16:24:46.103328Z"",""updated_at"":""2026-05-16T16:24:46.103328Z""}","{""payment_id"":2070688,""order_id"":2070688,""payment_method"":""card"",""amount"":300.0,""status"":""refunded"",""processor_ref"":null,""paid_at"":null,""created_at"":""2026-05-16T16:24:46.103328Z"",""updated_at"":""2026-05-16T16:24:51.899200Z""}",cdc.public.payments,2,3106,2026-05-16 16:24:52.729,2026-05-16 16:25:20.751
4,u,1778948691909,30107335024,5996472,payments,false,"{""payment_id"":2070693,""order_id"":2070693,""payment_method"":""wallet"",""amount"":365.56,""status"":""pending"",""processor_ref"":null,""paid_at"":null,""created_at"":""2026-05-16T16:24:46.335547Z"",""updated_at"":""2026-05-16T16:24:46.335547Z""}","{""payment_id"":2070693,""order_id"":2070693,""payment_method"":""wallet"",""amount"":365.56,""status"":""captured"",""processor_ref"":""ch_58snpvoiref9extttsl19yl1"",""paid_at"":""2026-05-16T16:24:51.908862Z"",""created_at"":""2026-05-16T16:24:46.335547Z"",""updated_at"":""2026-05-16T16:24:51.908972Z""}",cdc.public.payments,2,3107,2026-05-16 16:24:52.729,2026-05-16 16:25:20.751


In [300]:
spark.sql("""
        CREATE TABLE IF NOT EXISTS lake.silver.payments (
            payment_id           STRING,
            order_id             STRING,
            payment_method       STRING,
            amount               DECIMAL,
            status               STRING,
            processor_ref        STRING,
            paid_at              TIMESTAMP,
            created_at           TIMESTAMP,
            last_source_update_ts TIMESTAMP,
            last_refresh_date_ts TIMESTAMP
        )
        USING iceberg
        PARTITIONED BY (days(paid_at), status)
        TBLPROPERTIES (
            'format-version'                  = '2',
            'write.format.default'            = 'parquet',
            'write.parquet.compression-codec' = 'zstd',
            'write.target-file-size-bytes'    = '134217728'
        );
        """)

DataFrame[]

In [313]:
print_as_df(spark.sql("select * from lake.silver.payments"))

,payment_id,order_id,payment_method,amount,status,processor_ref,paid_at,created_at,last_source_update_ts,last_refresh_date_ts


In [ ]:
{"payment_id":2070686,"order_id":2070686,"payment_method":"card","amount":441.33,"status":"pending","processor_ref":null,"paid_at":null,"created_at":"2026-05-16T16:24:46.012176Z","updated_at":"2026-05-16T16:24:46.012176Z"}

In [295]:
payments_schema = StructType([
    StructField("payment_id", LongType(), False),
    StructField("order_id", LongType(), False),
    StructField("payment_method", StringType(), False),
    StructField("amount", FloatType(), False),
    StructField("status", StringType(), False),
    StructField("processor_ref", StringType(), True),
    StructField("paid_at", TimestampType(), True),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False),

])

In [314]:
def transforming_payments_to_silver(payments_schema, table_name = 'payments'):
    
    print(f"Start transforming \ntable name: {table_name}, table type: FACT, CDC? Yes, transforming type: MERGE")
    try:    
        
        last_watermark = get_last_watermark(table_name,
                                            layer='silver',
                                            prev_layer='bronze'
                                           )
    
        df_row_data_bronze_layer = fetch_bronze_layer_data(table_path_name = "lake.bronze."+table_name,
                                                            water_mark=last_watermark
                                                          )
        df_payments_flatern = flatten_kafka_payload(df = df_row_data_bronze_layer,
                                                       table_schema = payments_schema )

        window = Window.partitionBy("payment_id").orderBy(col("source_lsn").desc())
            
        df_payments_partitioned = df_payments_flatern.withColumn("rn", row_number().over(window)).filter(col("rn") == 1)
        print("  Is there new data in bronze layer?",not df_row_data_bronze_layer.isEmpty())
       
        if df_payments_partitioned.isEmpty():
            print(f"the bronze layer data is Empty!! for table {table_name} last water mark {last_watermark}")
            return 
        df_payments_partitioned = df_payments_partitioned.select(
                        "payment_id",
                        "order_id",
                        "payment_method",
                        "amount",
                        "status",
                        "processor_ref" ,
                        "paid_at",
                        "created_at",
                        "updated_at"     
            )
        df_payments_partitioned = df_payments_partitioned.withColumnRenamed("updated_at", "last_source_update_ts")
        df_payments_partitioned = df_payments_partitioned.withColumn("last_refresh_date_ts", current_timestamp())
        df_payments_partitioned.createOrReplaceTempView("silver_payment_updates")
        
        source = spark.table("silver_payment_updates").alias("s")
        target = spark.table("lake.silver.payments").alias("t")
    
        updates = (
            source
            .join(target, on="payment_id", how="left")
            .withColumn(
                "action",
                when(col("t.payment_id").isNotNull(), "update")
                .otherwise("insert")
            )
        )
    
        print("  Is there data after left join the bronze layer with silver layer?",not updates.isEmpty())
        updates.groupBy("action").count().show()
        
        spark.sql("""
            MERGE INTO lake.silver.payments od
            USING silver_payment_updates s
            ON od.payment_id = s.payment_id
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
            """)
        print(f" APPEND {table_name} data from bronze --> silver: successed")
        update_watermark_table(table_name = table_name)
        print("  water mark have been updated will be >", last_watermark)
    except Exceptiona as e : 
        raise Exception(e + f"\n transforming {table_name} from bronze --> silver: failed")
       

In [316]:
transforming_payments_to_silver(payments_schema=payments_schema)

Start transforming 
table name: payments, table type: FACT, CDC? Yes, transforming type: MERGE
  Is there new data in bronze layer? False
  No data to insert


In [317]:
# def mearge_orders_rows(df):
#     df_flat_u_op = df.withColumn(
#         "after_payload_value", from_json(col("after_payload"),orders_schema)
#                         ).select(
#                                 "*",
#                                 "after_payload_value.*",    
#                             ).filter(
#                                     (col("operation") != 'd') & (col("after_payload").isNotNull() 
#                                                                 )).drop("after_payload_value", "after_payload", "before_payload")
#     window = Window.partitionBy("order_id").orderBy(col("source_lsn").desc())
    
#     df_flat_u_op_partitioned = df_flat_u_op.withColumn("rn", row_number().over(window)).filter(col("rn") == 1).drop('rn')

#     if len(df_flat_u_op) = 0:
#         return 
#     df_flat_u_op_partitioned = df_flat_u_op_partitioned.select(
#                 "order_id",
#                 "customer_id",
#                 "restaurant_id",
#                 "driver_id",
#                 "pickup_zone_id",
#                 "dropoff_zone_id",
#                 "status",
#                 "subtotal" ,
#                 "delivery_fee",
#                 "service_fee",
#                 "discount",  
#                 "tip",
#                 "total",
#                 "placed_at",
#                 "confirmed_at",
#                 "ready_at",
#                 "picked_up_at",
#                 "delivered_at",
#                 "cancelled_at",
#                 "created_at",
#                 "updated_at"      
#     )
#     df_flat_u_op_partitioned = df_flat_u_op_partitioned.withColumnRenamed("updated_at", "last_source_update_ts")
#     df_flat_u_op_partitioned = df_flat_u_op_partitioned.withColumn("last_refresh_date_ts", current_timestamp())
#     df_flat_u_op_partitioned.createOrReplaceTempView("silver_orders_updates")
#     spark.sql("""
#         MERGE INTO lake.silver.orders od
#         USING silver_orders_updates s
#         ON od.order_id = s.order_id
#         WHEN MATCHED THEN UPDATE SET *
#         WHEN NOT MATCHED THEN INSERT *
#         """)
#     print_as_df(df_flat_u_op_partitioned)

# lake.silver.order_status_events processing 

In [346]:
spark.sql("""
            CREATE OR REPLACE TABLE lake.silver.order_status_events (
            event_id             STRING,
            order_id             STRING,
            from_status          STRING,
            to_status            STRING,
            event_ts             TIMESTAMP,
            actor_type           STRING,
            actor_id             STRING,
            notes                STRING,
            created_at           TIMESTAMP,
            last_refresh_date_ts TIMESTAMP
        )
        USING iceberg
        PARTITIONED BY (days(created_at), to_status)
        TBLPROPERTIES (
            'format-version'                  = '2',
            'write.format.default'            = 'parquet',
            'write.parquet.compression-codec' = 'zstd',
            'write.target-file-size-bytes'    = '134217728'
        );
        """)

DataFrame[]

In [336]:
order_status_events_schema = StructType([
    StructField("event_id", LongType(), False),
    StructField("order_id", LongType(), False),
    StructField("from_status", StringType(), True),
    StructField("to_status", StringType(), False),
    StructField("event_ts", TimestampType(), False),
    StructField("actor_type", StringType(), True),
    StructField("actor_id", StringType(), True),
    StructField("notes", StringType(), True),
    StructField("created_at", TimestampType(), False),

])

In [335]:
# {"event_id":3367386,"order_id":2066768,"from_status":null,"to_status":"placed","event_ts":"2026-05-16T12:53:05.206716Z","actor_type":"customer","actor_id":40075,"notes":null,"created_at":"2026-05-16T12:53:05.206958Z"}

In [55]:
df = spark.sql("SELECT * FROM lake.silver.order_status_events")
print_as_df(df)

,event_id,order_id,from_status,to_status,event_ts,actor_type,actor_id,notes,created_at,last_refresh_date_ts


In [360]:

def transforming_order_status_events_to_silver(schema, table_name = "order_status_events"):
    

    print(f"Start transforming \ntable name: {table_name}, table type: FACT, CDC? Yes, transforming type: APPEND")
    try:    
        
        last_watermark = get_last_watermark(table_name,
                                            layer='silver',
                                            prev_layer='bronze'
                                           )
    
        df_row_data_bronze_layer = fetch_bronze_layer_data(table_path_name = "lake.bronze."+table_name,
                                                            water_mark=last_watermark
                                                          )
        df_order_status_events_flatern = (df = df_row_data_bronze_layer,
                                                       table_schema = schema ).dropDuplicates(['event_id', 'source_lsn'])
        df_order_status_events_ids = spark.sql("""
                SELECT  oditm.event_id  
                FROM lake.silver.order_status_events oditm
             """
             )
    
        df_order_status_events_joined = df_order_status_events_flatern.join(
                                    df_order_status_events_ids,
                                    on="event_id", how='left_anti'
                                    )
        print("  Is there data in the silver layer?",not df_order_status_events_ids.isEmpty())
        print("  Is there data after left join the bronze layer with silver layer?",not df_order_status_events_joined.isEmpty())
        if df_order_status_events_joined.isEmpty():
            print(f"the bronze layer data is Empty!! for table {table_name} last water mark {last_watermark}")
            return 

                
        df_order_status_events_joined = df_order_status_events_joined.withColumn("last_refresh_date_ts", current_timestamp())
        df_order_status_events_joined = df_order_status_events_joined.select(
                                        "event_id",
                                        "order_id",
                                        "from_status",
                                        "to_status",
                                        "event_ts",
                                        "actor_type"  ,
                                        "actor_id",
                                        "notes",
                                        "created_at",
                                        "last_refresh_date_ts" 
        )#.cache() # use it if needed 
        df_order_status_events_joined.createOrReplaceTempView("order_status_events_apending")
        count_new_rows = df_order_status_events_joined.count()
        spark.sql("""
                  INSERT INTO lake.silver.order_status_events 
                  SELECT * FROM order_status_events_apending
                  """)
        #df_order_status_events_joined.unpersist()
    
        print(f"APPEND {table_name} data from bronze --> silver: successed")
        print("  total of new inserts",count_new_rows)
        update_watermark_table(table_name = table_name)
        print("  water mark have been updated will be >", last_watermark)
    except Exception as e : 
        raise Exception(e ,f"\n transforming {table_name} from bronze --> silver: failed")

In [348]:
run_func(transforming_order_status_events_to_silver, order_status_events_schema)

Start transforming 
table name: order_status_events, table type: FACT, CDC? Yes, transforming type: APPEND
  Is there data in the silver layer? False
  Is there data after left join the bronze layer with silver layer? True
APPEND order_status_events data from bronze --> silver: successed
  total of new inserts 52318
Updating the watermark table lake.silver.order_status_events Successed 
  water mark have been updated to  2026-05-16 14:19:20.970000
Duration: 7.830971 sec


# lake.silver.reviews processing 

In [67]:
df = spark.sql("SELECT * FROM lake.bronze.reviews")
print_as_df(df)

,operation,source_ts_ms,source_lsn,source_txid,source_table,source_snapshot,before_payload,after_payload,kafka_topic,kafka_partition,kafka_offset,kafka_timestamp,ingestion_ts
0,c,1779054890302,30212403816,6005694,reviews,false,None,"{""review_id"":136810,""order_id"":338269,""customer_id"":10168,""restaurant_id"":881,""driver_id"":5178,""food_rating"":4,""delivery_rating"":2,""comment"":null,""submitted_at"":""2026-05-17T21:54:50.282377Z"",""created_at"":""2026-05-17T21:54:50.282510Z""}",cdc.public.reviews,0,832,2026-05-17 21:54:50.953,2026-05-17 21:55:00.008
1,c,1779054890315,30212607544,6005706,reviews,false,None,"{""review_id"":136814,""order_id"":338942,""customer_id"":7638,""restaurant_id"":220,""driver_id"":14146,""food_rating"":4,""delivery_rating"":5,""comment"":null,""submitted_at"":""2026-05-17T21:54:50.314634Z"",""created_at"":""2026-05-17T21:54:50.314800Z""}",cdc.public.reviews,0,833,2026-05-17 21:54:50.954,2026-05-17 21:55:00.008
2,c,1779054890322,30212641480,6005709,reviews,false,None,"{""review_id"":136816,""order_id"":339464,""customer_id"":4229,""restaurant_id"":1287,""driver_id"":5513,""food_rating"":5,""delivery_rating"":5,""comment"":""والله لذيذ، تسلمون."",""submitted_at"":""2026-05-17T21:54:50.320402Z"",""created_at"":""2026-05-17T21:54:50.320533Z""}",cdc.public.reviews,0,834,2026-05-17 21:54:50.954,2026-05-17 21:55:00.008
3,c,1779054890324,30212662088,6005711,reviews,false,None,"{""review_id"":136817,""order_id"":338551,""customer_id"":6151,""restaurant_id"":741,""driver_id"":4407,""food_rating"":5,""delivery_rating"":4,""comment"":""Average meal, expected better for the price."",""submitted_at"":""2026-05-17T21:54:50.323862Z"",""created_at"":""2026-05-17T21:54:50.323972Z""}",cdc.public.reviews,0,835,2026-05-17 21:54:50.954,2026-05-17 21:55:00.008
4,c,1779054890328,30212691824,6005713,reviews,false,None,"{""review_id"":136818,""order_id"":338431,""customer_id"":1661,""restaurant_id"":1566,""driver_id"":4870,""food_rating"":5,""delivery_rating"":4,""comment"":null,""submitted_at"":""2026-05-17T21:54:50.326827Z"",""created_at"":""2026-05-17T21:54:50.326940Z""}",cdc.public.reviews,0,836,2026-05-17 21:54:50.954,2026-05-17 21:55:00.008


In [369]:
spark.sql("""
            CREATE OR REPLACE TABLE lake.silver.reviews (
            review_id            STRING,
            order_id             STRING,
            customer_id          STRING,
            restaurant_id        STRING,
            driver_id            STRING,
            food_rating          DECIMAL,
            delivery_rating      DECIMAL,
            comment              STRING,
            submitted_at         TIMESTAMP,
            last_refresh_date_ts TIMESTAMP
        )
        USING iceberg
        PARTITIONED BY (days(submitted_at))
        TBLPROPERTIES (
            'format-version'                  = '2',
            'write.format.default'            = 'parquet',
            'write.parquet.compression-codec' = 'zstd',
            'write.target-file-size-bytes'    = '134217728'
        );
        """)

DataFrame[]

In [ ]:
#{"review_id":136810,"order_id":338269,"customer_id":10168,"restaurant_id":881,"driver_id":5178,"food_rating":4,"delivery_rating":2,"comment":null,"submitted_at":"2026-05-17T21:54:50.282377Z","created_at":"2026-05-17T21:54:50.282510Z"}

In [361]:
reviews_schema = StructType([
    StructField("review_id", LongType(), False),
    StructField("order_id", LongType(), False),
    StructField("customer_id", StringType(), False),
    StructField("restaurant_id", StringType(), False),
    StructField("driver_id", StringType(), False),
    StructField("food_rating", FloatType(), False),
    StructField("delivery_rating", FloatType(), False),
    StructField("comment", StringType(), True),
    StructField("submitted_at", TimestampType(), False),
    StructField("last_refresh_date_ts", TimestampType(), False),
])

In [89]:
df = spark.sql(""" SELECT * FROM lake.silver.reviews""")
print_as_df(df)

,review_id,order_id,customer_id,restaurant_id,driver_id,food_rating,delivery_rating,comment,submitted_at,last_refresh_date_ts


In [372]:
def transforming_reviews_to_silver(schema, table_name = "reviews"):

    
    print(f"Start transforming \ntable name: {table_name}, table type: FACT, CDC? Yes, transforming type: APPEND")
    try:    
        
        last_watermark = get_last_watermark(table_name,
                                            layer='silver',
                                            prev_layer='bronze'
                                           )
    
        df_row_data_bronze_layer = fetch_bronze_layer_data(table_path_name = "lake.bronze."+table_name,
                                                            water_mark=last_watermark
                                                          )
        df_reviews_flatern = flatten_kafka_payload(df = df_row_data_bronze_layer,
                                                       table_schema = schema ).dropDuplicates(['review_id', 'source_lsn'])
        
        df_reviews_ids = spark.sql("""
                    SELECT  rv.review_id  
                    FROM lake.silver.reviews rv
                                 """
                                 )
        
        df_reviews_joined = df_reviews_flatern.join(
                                        df_reviews_ids,
                                        on="review_id", how='left_anti'
                                        )
        print("  Is there data in the silver layer?",not df_reviews_ids.isEmpty())
        print("  Is there data after left join the bronze layer with silver layer?",not df_reviews_joined.isEmpty())
        if df_reviews_joined.isEmpty():
            print(f"the bronze layer data is Empty!! for table {table_name} last water mark {last_watermark}")
            return 
        df_reviews_joined = df_reviews_joined.withColumn("last_refresh_date_ts", current_timestamp())
        df_reviews_joined = df_reviews_joined.select(
                                        "review_id",
                                        "order_id",
                                        "customer_id",
                                        "restaurant_id",
                                        "driver_id",
                                        "food_rating"  ,
                                        "delivery_rating",
                                        "comment",
                                        "submitted_at",
                                        "last_refresh_date_ts" 
        )#.cache()
        df_reviews_joined.createOrReplaceTempView("reviews_apending")
        count_new_rows = df_reviews_joined.count()
        spark.sql("""
                  INSERT INTO lake.silver.reviews 
                  SELECT * FROM reviews_apending
                  """)
       # df_reviews_joined.unpersist()    
        
        print(f"APPEND {table_name} data from bronze --> silver: successed")
        print("  total of new inserts",count_new_rows)
        update_watermark_table(table_name = table_name)
        print("  water mark have been updated will be >", last_watermark)
    except Exception as e : 
        raise Exception(e ,f"\n transforming {table_name} from bronze --> silver: failed")

In [371]:
run_func(transforming_reviews_to_silver, reviews_schema)

Start transforming 
table name: reviews, table type: FACT, CDC? Yes, transforming type: APPEND
  Is there data in the silver layer? False
  Is there data after left join the bronze layer with silver layer? True
APPEND reviews data from bronze --> silver: successed
  total of new inserts 4684
Updating the watermark table lake.silver.reviews Successed 
  water mark have been updated to  2026-05-16 14:19:20.970000
total of new insert row into lake.silver.reviews  4684
Duration: 5.243806 sec


# lake.silver.menu_items processing

In [ ]:
{"menu_item_id":7235,"restaurant_id":2049,"name":"Soft Drink","category":"drink","price":5.51,"is_available":true,"created_at":"2026-05-12T20:07:56.477834Z","updated_at":"2026-05-13T16:33:21.594729Z"}

In [33]:
spark.sql("""
            CREATE OR REPLACE TABLE   lake.silver.menu_items (
            menu_item_id            STRING,
            restaurant_id           STRING,
            name                    STRING,
            category                STRING,
            price                   DECIMAL(8,2),
            is_available            BOOLEAN,
            created_at_ts             TIMESTAMP,
            eff_start_ts       TIMESTAMP,
            eff_end_ts         TIMESTAMP,
            is_current              BOOLEAN,
            last_source_update_ts TIMESTAMP,
            last_refresh_date_ts    TIMESTAMP
        )
        USING iceberg
        TBLPROPERTIES (
            'format-version'                  = '2',
            'write.format.default'            = 'parquet',
            'write.parquet.compression-codec' = 'zstd',
            'write.target-file-size-bytes'    = '134217728'
        );
        """)
menu_items_schema = StructType([
    StructField("menu_item_id", LongType(), False),
    StructField("restaurant_id", LongType(), False),
    StructField("name", StringType(), False),
    StructField("category", StringType(), False),
    StructField("price", DecimalType(8,2), False),
    StructField("is_available", BooleanType(), False),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False)
])
# spark.sql("""
# CREATE OR REPLACE TABLE  lake.silver.watermarks (
#     table_name        STRING,
#     last_watermark    TIMESTAMP,
#     updated_at        TIMESTAMP
# )
# USING iceberg
#             """)

In [32]:
spark.sql("""
DELETE FROM lake.silver.watermarks
WHERE table_name = 'menu_items'
""")

DataFrame[]

In [111]:
# test cased for changing exist data from kafka 

In [38]:
a = {"menu_item_id":7235,"restaurant_id":2049,"name":"Hot Drink","category":"drink","price":6.551,"is_available":True,"created_at":"2026-05-12T20:07:56.477834Z","updated_at":str(datetime.now())}

In [40]:
df = spark.sql(""" SELECT * FROM lake.bronze.menu_items""")
print_as_df(df)

,operation,source_ts_ms,source_lsn,source_txid,source_table,source_snapshot,before_payload,after_payload,kafka_topic,kafka_partition,kafka_offset,kafka_timestamp,ingestion_ts
0,r,NaN,1,NaN,menu_items,menu_items,None,"{""menu_item_id"":1,""restaurant_id"":3000,""name"":""Milkshake"",""category"":""drink"",""price"":18.88,""is_available"":true,""created_at"":""2026-05-12T20:07:56.477Z"",""updated_at"":""2026-05-12T20:07:56.477Z""}",inital_load_menu_items,NaN,NaN,NaT,2026-06-17 03:18:08.358074
1,r,NaN,1,NaN,menu_items,menu_items,None,"{""menu_item_id"":2,""restaurant_id"":3000,""name"":""Soft Drink"",""category"":""drink"",""price"":9.24,""is_available"":true,""created_at"":""2026-05-12T20:07:56.477Z"",""updated_at"":""2026-05-12T20:07:56.477Z""}",inital_load_menu_items,NaN,NaN,NaT,2026-06-17 03:18:08.358074
2,r,NaN,1,NaN,menu_items,menu_items,None,"{""menu_item_id"":3,""restaurant_id"":3000,""name"":""Classic Burger"",""category"":""main"",""price"":28.56,""is_available"":true,""created_at"":""2026-05-12T20:07:56.477Z"",""updated_at"":""2026-05-12T20:07:56.477Z""}",inital_load_menu_items,NaN,NaN,NaT,2026-06-17 03:18:08.358074
3,r,NaN,1,NaN,menu_items,menu_items,None,"{""menu_item_id"":4,""restaurant_id"":3000,""name"":""Onion Rings"",""category"":""side"",""price"":17.61,""is_available"":true,""created_at"":""2026-05-12T20:07:56.477Z"",""updated_at"":""2026-05-12T20:07:56.477Z""}",inital_load_menu_items,NaN,NaN,NaT,2026-06-17 03:18:08.358074
4,r,NaN,1,NaN,menu_items,menu_items,None,"{""menu_item_id"":5,""restaurant_id"":3000,""name"":""Double Cheese Burger"",""category"":""main"",""price"":38.08,""is_available"":true,""created_at"":""2026-05-12T20:07:56.477Z"",""updated_at"":""2026-05-12T20:07:56.477Z""}",inital_load_menu_items,NaN,NaN,NaT,2026-06-17 03:18:08.358074


In [42]:
df_test = df.filter(col("source_lsn") ==  "1").withColumn("after_payload", lit(json.dumps(a))).withColumn("ingestion_ts", to_timestamp(lit(datetime.now())))
print_as_df(df_test)

,operation,source_ts_ms,source_lsn,source_txid,source_table,source_snapshot,before_payload,after_payload,kafka_topic,kafka_partition,kafka_offset,kafka_timestamp,ingestion_ts
0,r,NaN,1,NaN,menu_items,menu_items,None,"{""menu_item_id"": 7235, ""restaurant_id"": 2049, ""name"": ""Hot Drink"", ""category"": ""drink"", ""price"": 6.551, ""is_available"": true, ""created_at"": ""2026-05-12T20:07:56.477834Z"", ""updated_at"": ""2026-06-27 02:07:04.036045""}",inital_load_menu_items,NaN,NaN,NaT,2026-06-27 02:08:24.332257
1,r,NaN,1,NaN,menu_items,menu_items,None,"{""menu_item_id"": 7235, ""restaurant_id"": 2049, ""name"": ""Hot Drink"", ""category"": ""drink"", ""price"": 6.551, ""is_available"": true, ""created_at"": ""2026-05-12T20:07:56.477834Z"", ""updated_at"": ""2026-06-27 02:07:04.036045""}",inital_load_menu_items,NaN,NaN,NaT,2026-06-27 02:08:24.332257
2,r,NaN,1,NaN,menu_items,menu_items,None,"{""menu_item_id"": 7235, ""restaurant_id"": 2049, ""name"": ""Hot Drink"", ""category"": ""drink"", ""price"": 6.551, ""is_available"": true, ""created_at"": ""2026-05-12T20:07:56.477834Z"", ""updated_at"": ""2026-06-27 02:07:04.036045""}",inital_load_menu_items,NaN,NaN,NaT,2026-06-27 02:08:24.332257
3,r,NaN,1,NaN,menu_items,menu_items,None,"{""menu_item_id"": 7235, ""restaurant_id"": 2049, ""name"": ""Hot Drink"", ""category"": ""drink"", ""price"": 6.551, ""is_available"": true, ""created_at"": ""2026-05-12T20:07:56.477834Z"", ""updated_at"": ""2026-06-27 02:07:04.036045""}",inital_load_menu_items,NaN,NaN,NaT,2026-06-27 02:08:24.332257
4,r,NaN,1,NaN,menu_items,menu_items,None,"{""menu_item_id"": 7235, ""restaurant_id"": 2049, ""name"": ""Hot Drink"", ""category"": ""drink"", ""price"": 6.551, ""is_available"": true, ""created_at"": ""2026-05-12T20:07:56.477834Z"", ""updated_at"": ""2026-06-27 02:07:04.036045""}",inital_load_menu_items,NaN,NaN,NaT,2026-06-27 02:08:24.332257


In [43]:
def transforming_menu_items_to_silver(schema, table_name = 'menu_items'):
    
    print(f"Start transforming \ntable name: {table_name}, table type: DIM: SCD2, CDC? Yes, transforming type: MERGE")
    try:    
        
        last_watermark = get_last_watermark(table_name,
                                            layer='silver',
                                            prev_layer='bronze'
                                           )
       
        df_row_data_bronze_layer = fetch_bronze_layer_data(table_path_name = "lake.bronze."+table_name,
                                                            water_mark=last_watermark
                                                          )
        df_menu_items_flatern = flatten_kafka_payload(df = df_row_data_bronze_layer,
                                                       table_schema = schema )
        
        # remove the duplicates rows 
        window = Window.partitionBy("menu_item_id").orderBy(col("source_lsn").desc())        
        df_cdc = df_menu_items_flatern.withColumn("rn", row_number().over(window)).filter(col("rn") == 1).drop('rn')
        #print_as_df(df_cdc)
        df_silver_current = spark.sql("SELECT * FROM lake.silver.menu_items WHERE is_current = true")
        df_changed = df_cdc.join(
            df_silver_current,
            on="menu_item_id",
            how="left"
        ).filter(
           df_silver_current.menu_item_id.isNull() |
            (df_cdc.updated_at != df_silver_current.last_source_update_ts)
        ).select(df_cdc["*"])#.cache()
        
        df_changed.createOrReplaceTempView("menu_items_merge")
        print("  Is there Caputred data from Kafka?",not df_cdc.isEmpty())
        print("  Is there data in the silver layer?",not df_silver_current.isEmpty())
        print("  Is there data after left join df_cdc with df_silver_current?",not df_changed.isEmpty())
        new_rows = df_changed.count()
        try:
            if  df_changed.isEmpty():
                print("  No data in df_canged to Upsert")
                return

            spark.sql("""
                        MERGE INTO lake.silver.menu_items mitm
                        USING menu_items_merge mitm_updated
                             on mitm.menu_item_id = mitm_updated.menu_item_id
                             AND mitm.is_current = True
                        WHEN MATCHED THEN UPDATE SET 
                                            mitm.is_current = False,
                                            mitm.eff_end_ts = mitm_updated.updated_at,
                                            mitm.last_source_update_ts = mitm_updated.updated_at
                        """)
        
            spark.sql("""
                        INSERT INTO lake.silver.menu_items SELECT 
                                        menu_item_id,
                                        restaurant_id,
                                        name,
                                        category,
                                        price,
                                        is_available,
                                        created_at,
                                        mitm_updated.updated_at,
                                        NULL,
                                        True,
                                        updated_at,
                                        CURRENT_TIMESTAMP  
                                    FROM menu_items_merge mitm_updated              
                        """)
        
        except Exception as e:
            raise Exception("Error occred in function merge the menue items rows", e)
        
        print(f"MERGE {table_name} data from bronze --> silver: successed")
        print("  total of new inserts",new_rows)
        update_watermark_table(table_name = table_name)
        print("  water mark have been updated will be >", last_watermark)
    except Exception as e : 
        raise Exception(e ,f"\n transforming {table_name} from bronze --> silver: failed")

In [35]:
run_func(transforming_menu_items_to_silver, menu_items_schema)

Start transforming 
table name: menu_items, table type: DIM: SCD2, CDC? Yes, transforming type: MERGE
  Is there Caputred data from Kafka? True
  Is there data in the silver layer? False
  Is there data after left join df_cdc with df_silver_current? True
MERGE menu_items data from bronze --> silver: successed
  total of new inserts 22794
Updating the watermark table lake.silver.menu_items Successed 
  water mark have been updated will be > 2026-06-17 03:18:08.358074
Duration: 11.053999 sec


In [36]:
df = spark.sql(""" SELECT * FROM lake.silver.menu_items WHERE menu_item_id = '7235' """)
print_as_df(df)

,menu_item_id,restaurant_id,name,category,price,is_available,created_at_ts,eff_start_ts,eff_end_ts,is_current,last_source_update_ts,last_refresh_date_ts
0,7235,2049,Soft Drink,drink,4.42,True,2026-05-12 20:07:56.477,2026-05-16 16:23:05.199,NaT,True,2026-05-16 16:23:05.199,2026-06-27 02:04:31.922480


# lake.silver.drivers processing

In [205]:
df = spark.sql("SELECT * FROM lake.bronze.drivers")
print_as_df(df)

,operation,source_ts_ms,source_lsn,source_txid,source_table,source_snapshot,before_payload,after_payload,kafka_topic,kafka_partition,kafka_offset,kafka_timestamp,ingestion_ts
0,u,1779055370229,30458181272,6037608,drivers,false,None,"{""driver_id"":3803,""full_name"":""Khalid Al-Hejailan"",""phone"":""+966524848120"",""vehicle_type"":""car"",""city_id"":5,""status"":""available"",""onboarded_at"":""2025-09-27T13:48:25.516150Z"",""is_active"":true,""created_at"":""2026-05-12T20:07:56.477834Z"",""updated_at"":""2026-05-17T22:02:50.228011Z""}",cdc.public.drivers,0,483,2026-05-17 22:02:50.745,2026-05-17 22:04:08.515
1,u,1779055375284,30460756704,6037911,drivers,false,None,"{""driver_id"":5500,""full_name"":""Maha Al-Faifi"",""phone"":""+966596122727"",""vehicle_type"":""scooter"",""city_id"":5,""status"":""busy"",""onboarded_at"":""2025-11-04T22:00:55.525988Z"",""is_active"":true,""created_at"":""2026-05-12T20:07:56.477834Z"",""updated_at"":""2026-05-17T22:02:55.282508Z""}",cdc.public.drivers,0,484,2026-05-17 22:02:55.758,2026-05-17 22:04:08.515
2,u,1779055375305,30460797304,6037917,drivers,false,None,"{""driver_id"":10773,""full_name"":""Zahra Al-Tamimi"",""phone"":""+966596602148"",""vehicle_type"":""bike"",""city_id"":1,""status"":""busy"",""onboarded_at"":""2026-04-10T00:36:21.549460Z"",""is_active"":true,""created_at"":""2026-05-12T20:07:56.477834Z"",""updated_at"":""2026-05-17T22:02:55.304347Z""}",cdc.public.drivers,0,485,2026-05-17 22:02:55.758,2026-05-17 22:04:08.515
3,u,1779055380460,30464166264,6038275,drivers,false,None,"{""driver_id"":13177,""full_name"":""Rana Al-Zahrani"",""phone"":""+966570507844"",""vehicle_type"":""scooter"",""city_id"":1,""status"":""busy"",""onboarded_at"":""2024-09-21T22:46:07.559989Z"",""is_active"":true,""created_at"":""2026-05-12T20:07:56.477834Z"",""updated_at"":""2026-05-17T22:03:00.459911Z""}",cdc.public.drivers,0,486,2026-05-17 22:03:00.795,2026-05-17 22:04:08.515
4,u,1779055380474,30464174664,6038276,drivers,false,None,"{""driver_id"":14144,""full_name"":""Mishal Al-Yami"",""phone"":""+966583968484"",""vehicle_type"":""scooter"",""city_id"":1,""status"":""available"",""onboarded_at"":""2024-11-29T00:02:51.563943Z"",""is_active"":true,""created_at"":""2026-05-12T20:07:56.477834Z"",""updated_at"":""2026-05-17T22:03:00.473346Z""}",cdc.public.drivers,0,487,2026-05-17 22:03:00.795,2026-05-17 22:04:08.515


In [206]:
a= {"driver_id":1669,"full_name":"Mishal Al-Harbi","phone":"+966551952999","vehicle_type":"car","city_id":3,"status":"offline","onboarded_at":"2025-07-18T12:58:17.506058Z","is_active":True,"created_at":"2026-05-12T20:07:56.477834Z","updated_at":"2026-05-16T12:53:05.510185Z"}

In [207]:
df_test = df.filter(col("source_lsn") ==  "29951188016").withColumn("after_payload", lit(json.dumps(a))).withColumn("ingestion_ts", to_timestamp(lit(datetime.now())))
print_as_df(df_test)

,operation,source_ts_ms,source_lsn,source_txid,source_table,source_snapshot,before_payload,after_payload,kafka_topic,kafka_partition,kafka_offset,kafka_timestamp,ingestion_ts
0,u,1778935985513,29951188016,5979347,drivers,false,None,"{""driver_id"": 1669, ""full_name"": ""Mishal Al-Harbi"", ""phone"": ""+966551952999"", ""vehicle_type"": ""car"", ""city_id"": 3, ""status"": ""offline"", ""onboarded_at"": ""2025-07-18T12:58:17.506058Z"", ""is_active"": true, ""created_at"": ""2026-05-12T20:07:56.477834Z"", ""updated_at"": ""2026-05-16T12:53:05.510185Z""}",cdc.public.drivers,0,0,2026-05-16 12:53:07.327,2026-05-25 18:49:23.270459
1,u,1778935985513,29951188016,5979347,drivers,false,None,"{""driver_id"": 1669, ""full_name"": ""Mishal Al-Harbi"", ""phone"": ""+966551952999"", ""vehicle_type"": ""car"", ""city_id"": 3, ""status"": ""offline"", ""onboarded_at"": ""2025-07-18T12:58:17.506058Z"", ""is_active"": true, ""created_at"": ""2026-05-12T20:07:56.477834Z"", ""updated_at"": ""2026-05-16T12:53:05.510185Z""}",cdc.public.drivers,0,0,2026-05-16 12:53:07.327,2026-05-25 18:49:23.270459
2,u,1778935985513,29951188016,5979347,drivers,false,None,"{""driver_id"": 1669, ""full_name"": ""Mishal Al-Harbi"", ""phone"": ""+966551952999"", ""vehicle_type"": ""car"", ""city_id"": 3, ""status"": ""offline"", ""onboarded_at"": ""2025-07-18T12:58:17.506058Z"", ""is_active"": true, ""created_at"": ""2026-05-12T20:07:56.477834Z"", ""updated_at"": ""2026-05-16T12:53:05.510185Z""}",cdc.public.drivers,0,0,2026-05-16 12:53:07.327,2026-05-25 18:49:23.270459
3,u,1778935985513,29951188016,5979347,drivers,false,None,"{""driver_id"": 1669, ""full_name"": ""Mishal Al-Harbi"", ""phone"": ""+966551952999"", ""vehicle_type"": ""car"", ""city_id"": 3, ""status"": ""offline"", ""onboarded_at"": ""2025-07-18T12:58:17.506058Z"", ""is_active"": true, ""created_at"": ""2026-05-12T20:07:56.477834Z"", ""updated_at"": ""2026-05-16T12:53:05.510185Z""}",cdc.public.drivers,0,0,2026-05-16 12:53:07.327,2026-05-25 18:49:23.270459


In [36]:
spark.sql("""
            CREATE OR REPLACE TABLE   lake.silver.drivers (
            driver_id            STRING,
            full_name            STRING,
            phone                STRING,
            prev_vehicle_type    STRING,
            vehicle_type         STRING,
            prev_city_id         STRING,
            city_id              STRING,
            status               STRING,
            onboarded_at_ts         TIMESTAMP,
            is_active            BOOLEAN,
            created_at_ts           TIMESTAMP,
            last_refresh_ts TIMESTAMP,
            last_source_update_ts TIMESTAMP
        )
        USING iceberg
        PARTITIONED BY (city_id)
        TBLPROPERTIES (
            'format-version'                  = '2',
            'write.format.default'            = 'parquet',
            'write.parquet.compression-codec' = 'zstd',
            'write.target-file-size-bytes'    = '134217728'
        );
        """)
drivers_items_schema = StructType([
    StructField("driver_id", StringType(), False),
    StructField("full_name", StringType(), False),
    StructField("phone", StringType(), False),
    StructField("vehicle_type", StringType(), False),
    StructField("city_id", LongType(), False),
    StructField("status", StringType(), False),
    StructField("is_active", BooleanType(), False),
    StructField("onboarded_at", TimestampType(), False),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False)

])
# spark.sql("""
# CREATE OR REPLACE TABLE  lake.silver.watermarks (
#     table_name        STRING,
#     last_watermark    TIMESTAMP,
#     updated_at        TIMESTAMP
# )
# USING iceberg
#             """)

In [42]:
df = spark.sql(""" SELECT * FROM lake.silver.drivers  """)
print_as_df(df)

,driver_id,full_name,phone,prev_vehicle_type,vehicle_type,prev_city_id,city_id,status,onboarded_at_ts,is_active,created_at_ts,last_refresh_ts,last_source_update_ts
0,10003,Tala Al-Juhani,+966546693793,None,scooter,None,1,busy,2025-06-01 20:57:21.546,True,2026-05-12 20:07:56.477,2026-06-22 14:25:18.477566,2026-05-13 18:21:58.694
1,10007,Majed Al-Ruwais,+966575467314,None,car,None,1,available,2025-07-23 09:51:05.546,True,2026-05-12 20:07:56.477,2026-06-22 14:25:18.477566,2026-05-13 21:49:21.504
2,10008,Rawan Al-Sulaiman,+966525151769,None,bike,None,1,offline,2026-01-18 04:15:41.546,True,2026-05-12 20:07:56.477,2026-06-22 14:25:18.477566,2026-05-13 21:46:25.529
3,10009,Majed Al-Juhani,+966530240523,None,scooter,None,1,busy,2025-12-15 07:19:24.546,True,2026-05-12 20:07:56.477,2026-06-22 14:25:18.477566,2026-05-13 21:11:21.558
4,10010,Najla Al-Faraj,+966567694188,None,scooter,None,1,offline,2026-02-04 06:39:27.546,True,2026-05-12 20:07:56.477,2026-06-22 14:25:18.477566,2026-05-13 19:04:21.548


In [38]:
spark.sql("DELETE FROM lake.silver.watermarks WHERE table_name = 'drivers'")

DataFrame[]

In [39]:
def transforming_drivers_to_silver(schema, table_name = 'drivers'):

    print(f"Start transforming \ntable name: {table_name}, table type: DIM: SCD3, CDC? Yes, transforming type: MERGE")
    try:    
        
        last_watermark = get_last_watermark(table_name,
                                            layer='silver',
                                            prev_layer='bronze'
                                           )
       
        df_row_data_bronze_layer = fetch_bronze_layer_data(table_path_name = "lake.bronze."+table_name,
                                                            water_mark=last_watermark
                                                          )
        df_menu_items_flatern = flatten_kafka_payload(df = df_row_data_bronze_layer,
                                                       table_schema = schema )
        # remove the duplicates rows 
        window = Window.partitionBy("driver_id").orderBy(col("source_lsn").desc())        
        df_cdc_after = df_menu_items_flatern.withColumn("rn", row_number().over(window)).filter(col("rn") == 1).drop('rn')
        
        print("Is there Captured data from kafka?",not df_cdc_after.isEmpty())
        try:
            if  df_cdc_after.isEmpty():
                print("No data in df_canged to Upsert")
                return 
            df_cdc_after.createOrReplaceTempView("drivers_updates")
            
            spark.sql("""
                            MERGE INTO lake.silver.drivers dr
                            USING drivers_updates dru
                                ON dr.driver_id = dru.driver_id
                            WHEN MATCHED AND (
                                                    dr.city_id != dru.city_id OR
                                                    dr.vehicle_type != dru.vehicle_type OR
                                                    dr.status != dru.status OR
                                                    dr.phone != dru.phone OR
                                                    dr.is_active != dru.is_active
                                                ) THEN
                                UPDATE SET
                                   full_name = dru.full_name,
                                   phone = dru.phone,
                                   vehicle_type = dru.vehicle_type,
                                   city_id = dru.city_id,
                                   status = dru.status,
                                   is_active = dru.is_active,
                                   onboarded_at_ts = dru.onboarded_at,
                                   created_at_ts =  dru.created_at,
                                   prev_vehicle_type = CASE WHEN dr.vehicle_type != dru.vehicle_type THEN dr.vehicle_type ELSE dr.prev_vehicle_type END ,
                                   prev_city_id = CASE WHEN dr.city_id != dru.city_id THEN dr.city_id ELSE dr.prev_city_id END,
                                   last_source_update_ts = dru.updated_at,
                                   last_refresh_ts = CURRENT_TIMESTAMP
                           WHEN NOT MATCHED THEN
                               INSERT (
                                        driver_id,
                                        full_name,
                                        phone,
                                        vehicle_type,
                                        city_id,
                                        status,
                                        is_active,
                                        onboarded_at_ts,
                                        created_at_ts,
                                        prev_vehicle_type,
                                        prev_city_id,
                                        last_refresh_ts,
                                        last_source_update_ts
                                    )
                                    VALUES (
                                        dru.driver_id,
                                        dru.full_name,
                                        dru.phone,
                                        dru.vehicle_type,
                                        dru.city_id,
                                        dru.status,
                                        dru.is_active,
                                        dru.onboarded_at,
                                        dru.created_at,
                                        NULL,
                                        NULL,
                                        CURRENT_TIMESTAMP,
                                        dru.updated_at
                                    )
                                                            
                """)
            
        except Exception as e : 
                raise Exception(e ,f"could not UPSERT the data for table {table_name}")
        print(f"MERGE {table_name} data from bronze --> silver: successed")
        print("  total of new inserts",df_cdc_after.count())
        update_watermark_table(table_name = table_name)
        print("  water mark have been updated will be >", last_watermark)
    except Exception as e : 
        raise Exception(e ,f"\n transforming {table_name} from bronze --> silver: failed")

In [40]:
run_func(transforming_drivers_to_silver, drivers_items_schema)

Start transforming 
table name: drivers, table type: DIM: SCD3, CDC? Yes, transforming type: MERGE
Is there Captured data from kafka? True
MERGE drivers data from bronze --> silver: successed
  total of new inserts 15000
Updating the watermark table lake.silver.drivers Successed 
  water mark have been updated will be > 2026-06-17 03:18:02.625195
Duration: 4.931629 sec


In [464]:
# df = spark.sql(""" SELECT * FROM lake.silver.drivers where driver_id=1669 """)
# print_as_df(df)